# Artilharia e algumas estatísticas dos jogadores
## Etapa de construção de tabela gold

Agrupado dos artilheiros por jogador e time. Gols contras (o.g.) são desconsiderados.

In [0]:
from pyspark.sql.functions import *
from datetime import datetime, timedelta
from pyspark.sql.window import Window

In [0]:
df_goals = spark.read.table("apifootball.silver.goals").filter(~col("player_name").like("%o.g.%")) # remove o.g. goals
df_cards = spark.read.table("apifootball.silver.cards")

In [0]:
df_scorers = (
    df_goals
    .groupBy("player_name", "team_id","team_name")
    .agg(count("*").alias("goals"))
     .select(
         "player_name",
         "team_id",
         "team_name",
         "goals",
     )
).orderBy(col("goals").desc())

In [0]:
df_cards_agg = (
    df_cards.groupBy("team_id", "team_name", "player_name")
    .agg(
        sum(when(col("card_type") == "yellow card", 1).otherwise(0)).alias("yellow_cards"),
        sum(when(col("card_type") == "red card", 1).otherwise(0)).alias("red_cards"),
        count("*").alias("qt_cards")
    )
)

In [0]:
df_union = (
    df_goals.select("player_name", "team_id","team_name")
    .union(df_cards.select("player_name", "team_id","team_name"))
).distinct()

In [0]:
window_spec = Window.orderBy(desc("goals"))

df_result = (
    df_union.alias("u")
    .join(df_cards_agg.alias("c"),
          (col("u.player_name") == col("c.player_name")) & (col("u.team_id") == col("c.team_id")), "left")
    .join(df_scorers.alias("s"),
          (col("u.player_name") == col("s.player_name")) & (col("u.team_id") == col("s.team_id")), "left")
    .select(
        rank().over(window_spec).alias("rank"),
        col("u.player_name"),
        col("u.team_id"),
        col("u.team_name"),
        col("s.goals"),
        col("c.yellow_cards"),
        col("c.red_cards"),
        col("c.qt_cards"),
        expr('current_timestamp() - INTERVAL 3 HOURS').alias('datetime_processing_br')
    )
).orderBy("rank")

In [0]:
df_result.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("apifootball.gold.scorers")

In [0]:
spark.read.table("apifootball.gold.scorers").display()